In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# Universe
TICKERS = [
    "XLK", "XLY", "XLE", "XLF", "XLI",
    "XLB", "XLP", "XLV", "XLU", "GLD",
    "SHY", "SPY"
]

def download_raw_data(tickers, start="2005-01-01", end=None):
    """
    Download daily OHLCV + dividends for multiple tickers.
    Returns a long-format DataFrame.
    """
    if end is None:
        end = datetime.today().strftime("%Y-%m-%d")

    print(f"Downloading data from {start} to {end} ...")

    # Download with actions (includes Dividends and Stock Splits)
    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=False,          # keep raw + Adj Close
        actions=True,               # include Dividends
        group_by="ticker",
        threads=True,
        progress=True
    )

    all_data = []

    for ticker in tickers:
        try:
            df = raw[ticker].copy()
            df = df.reset_index()
            df["Ticker"] = ticker

            # Standardize column names
            df = df.rename(columns={
                "Date": "Date",
                "Open": "Open",
                "High": "High",
                "Low": "Low",
                "Close": "Close",
                "Adj Close": "Adj_Close",
                "Volume": "Volume",
                "Dividends": "Dividends",
                "Stock Splits": "Stock_Splits"
            })

            # Keep only useful columns
            cols = ["Date", "Ticker", "Open", "High", "Low", "Close",
                    "Adj_Close", "Volume", "Dividends", "Stock_Splits"]
            df = df[[c for c in cols if c in df.columns]]

            all_data.append(df)
            print(f"  ✓ {ticker}: {len(df)} rows")

        except Exception as e:
            print(f"  ✗ Failed to process {ticker}: {e}")

    if not all_data:
        raise ValueError("No data downloaded.")

    long_df = pd.concat(all_data, ignore_index=True)
    return long_df

def clean_and_align(df):
    """
    Clean missing values and unify trading dates across all tickers.
    """
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    # Sort
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

    # Create master calendar (union of all trading days)
    all_dates = sorted(df["Date"].unique())
    master_calendar = pd.DataFrame({"Date": all_dates})

    cleaned_list = []

    for ticker, group in df.groupby("Ticker"):
        g = group.set_index("Date")

        # Reindex to master calendar
        g = g.reindex(all_dates)

        # Forward fill price-related columns (reasonable for prices)
        price_cols = ["Open", "High", "Low", "Close", "Adj_Close"]
        g[price_cols] = g[price_cols].ffill()

        # Volume and Dividends: fill 0 (no trade / no dividend)
        g["Volume"] = g["Volume"].fillna(0)
        if "Dividends" in g.columns:
            g["Dividends"] = g["Dividends"].fillna(0)
        if "Stock_Splits" in g.columns:
            g["Stock_Splits"] = g["Stock_Splits"].fillna(0)

        g["Ticker"] = ticker
        g = g.reset_index()
        cleaned_list.append(g)

    cleaned = pd.concat(cleaned_list, ignore_index=True)

    # Drop rows that still have no Close price (very early dates)
    cleaned = cleaned.dropna(subset=["Close"])

    # Final sort
    cleaned = cleaned.sort_values(["Date", "Ticker"]).reset_index(drop=True)

    return cleaned

def get_data_summary(df):
    """Print summary of available dates and coverage."""
    print("\n=== Data Summary ===")
    summary = df.groupby("Ticker")["Date"].agg(["min", "max", "count"])
    summary.columns = ["First_Date", "Last_Date", "N_Days"]
    print(summary.to_string())
    print(f"\nTotal rows: {len(df)}")
    print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

def load_market_data(start="2005-01-01", end=None, save_path="clean_prices.csv"):
    """
    Main function: download + clean + optionally save.
    Returns the cleaned DataFrame.
    """
    raw = download_raw_data(TICKERS, start=start, end=end)
    clean = clean_and_align(raw)
    get_data_summary(clean)

    if save_path:
        clean.to_csv(save_path, index=False)
        print(f"\nSaved cleaned data to: {save_path}")

    return clean

# ============================================================
# Run when executed directly
# ============================================================
if __name__ == "__main__":
    df = load_market_data(
        start="2005-01-01",
        save_path="clean_prices.csv"
    )
    print("\nSample of the data:")
    print(df.head(10))

In [ ]:
import pandas as pd
import numpy as np

# Approximate trading days for each horizon
LOOKBACKS = {
    "R3m": 63,    # ~3 months
    "R6m": 126,   # ~6 months
    "R12m": 252   # ~12 months
}

def load_clean_data(path="clean_prices.csv"):
    """Load the cleaned price data from data_loader.py"""
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    return df

def calculate_total_returns(df, price_col="Adj_Close"):
    """
    Calculate 3m / 6m / 12m total returns using adjusted close.
    Returns are simple percentage returns.
    """
    for name, periods in LOOKBACKS.items():
        df[name] = df.groupby("Ticker")[price_col].pct_change(periods=periods)
    return df

def calculate_momentum(df):
    """
    Momentum Score = equal-weighted average of R3m, R6m, R12m
    """
    df["Momentum"] = (df["R3m"] + df["R6m"] + df["R12m"]) / 3
    return df

def calculate_moving_average(df, window=200, price_col="Adj_Close"):
    """200-day Simple Moving Average"""
    df["MA_200"] = df.groupby("Ticker")[price_col].transform(
        lambda x: x.rolling(window=window, min_periods=window).mean()
    )
    return df

def calculate_historical_volatility(df, window=60, price_col="Adj_Close"):
    """
    60-day Historical Volatility (annualized).
    Based on daily returns of Adj_Close.
    """
    df["Daily_Return"] = df.groupby("Ticker")[price_col].pct_change()
    df["HV_60"] = df.groupby("Ticker")["Daily_Return"].transform(
        lambda x: x.rolling(window=window, min_periods=window).std() * np.sqrt(252)
    )
    return df

def calculate_atr(df, window=14):
    """
    14-day Average True Range (simple rolling mean of True Range).
    True Range = max(High-Low, |High-PrevClose|, |Low-PrevClose|)
    """
    # Previous close (within each ticker)
    df["Prev_Close"] = df.groupby("Ticker")["Close"].shift(1)

    # True Range components
    tr1 = df["High"] - df["Low"]
    tr2 = (df["High"] - df["Prev_Close"]).abs()
    tr3 = (df["Low"] - df["Prev_Close"]).abs()

    df["TR"] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    # 14-day ATR
    df["ATR_14"] = df.groupby("Ticker")["TR"].transform(
        lambda x: x.rolling(window=window, min_periods=window).mean()
    )

    # Clean temporary columns
    df = df.drop(columns=["Prev_Close", "TR"], errors="ignore")
    return df

def add_all_indicators(df):
    """Run all indicator calculations in sequence"""
    print("Calculating total returns (3m / 6m / 12m)...")
    df = calculate_total_returns(df)

    print("Calculating Momentum Score...")
    df = calculate_momentum(df)

    print("Calculating 200-day Moving Average...")
    df = calculate_moving_average(df)

    print("Calculating 60-day Historical Volatility...")
    df = calculate_historical_volatility(df)

    print("Calculating 14-day ATR...")
    df = calculate_atr(df)

    return df

def select_output_columns(df):
    """Keep the most useful columns for the final market_data.csv"""
    keep = [
        "Date", "Ticker",
        "Open", "High", "Low", "Close", "Adj_Close", "Volume", "Dividends",
        "R3m", "R6m", "R12m", "Momentum",
        "MA_200", "HV_60", "ATR_14"
    ]
    # Only keep columns that actually exist
    keep = [c for c in keep if c in df.columns]
    return df[keep].copy()

def save_market_data(df, path="market_data.csv"):
    """Save the final dataset"""
    df.to_csv(path, index=False)
    print(f"\nSaved final data to: {path}")
    print(f"Shape: {df.shape}")

def get_indicator_summary(df):
    """Show when indicators become available (warm-up period)"""
    print("\n=== Indicator Warm-up Summary (first non-null date) ===")
    indicators = ["R3m", "R6m", "R12m", "Momentum", "MA_200", "HV_60", "ATR_14"]
    for ind in indicators:
        if ind in df.columns:
            first_valid = df.dropna(subset=[ind]).groupby("Ticker")["Date"].min()
            print(f"\n{ind}:")
            print(first_valid.to_string())

# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    # 1. Load cleaned prices
    df = load_clean_data("clean_prices.csv")
    print(f"Loaded clean data: {df.shape}")

    # 2. Calculate all indicators
    df = add_all_indicators(df)

    # 3. Select final columns
    final_df = select_output_columns(df)

    # 4. Show warm-up info
    get_indicator_summary(final_df)

    # 5. Save
    save_market_data(final_df, "market_data.csv")

    print("\nSample of final data (latest rows):")
    print(final_df.tail(12))